# Where do the letters go?

Part 1 of the Lovestats sequence-prediction series. Start with the smallest useful memory: one letter. Estimate a distribution over its successors, then follow the most likely successor repeatedly. The opening question is whether paths that meet ever separate under this rule. Later experiments can change the context length, introduce recurrent models, and finally explore LLM next-token prediction. Here each state is a letter.

Run all cells with the notebook directory as the working directory. Only the Python standard library and the adjacent CSV are required.

## Count adjacent letters within words

The input contains the 2-gram and */* (all word lengths and positions) columns from a [CSV copy](https://github.com/timelyportfolio/sunburstR/blob/master/inst/examples/ngrams2.csv), Git blob bac34b65ea3c8380939e25d6195a12667c0ce524. [Peter Norvig’s source and methodology](https://norvig.com/mayzner.html) give the corpus total and number of observed pairs. A missing pair has zero observed count. Word endings contribute no outgoing transition.

In [1]:
import csv
import json
import math
from pathlib import Path
from string import ascii_uppercase as alphabet

with Path("bigram_counts.csv").open() as source:
    rows = list(csv.DictReader(source))
counts = {r["pair"]: int(r["count"]) for r in rows}
assert len(rows) == len(counts) == 669
assert all(len(p) == 2 and all(c in alphabet for c in p) and n > 0 for p, n in counts.items())
assert sum(counts.values()) == 2_819_662_855_499
print(f"{len(counts)} observed pairs, {sum(counts.values()):,} occurrences")

669 observed pairs, 2,819,662,855,499 occurrences


## Estimate each next-letter distribution

Divide each pair count by the total outgoing count for its first letter. Rank by integer count and break ties alphabetically. A strongest edge represents the greedy choice. Its probability measures how much evidence the discarded alternatives carry collectively through the complement.

In [2]:
letters = {}
for start in alphabet:
    row = {end: counts.get(start + end, 0) for end in alphabet}
    total = sum(row.values())
    ranked = [{"letter": end, "count": row[end], "probability": row[end] / total}
              for end in sorted(row, key=lambda end: (-row[end], end))]
    assert math.isclose(sum(item["probability"] for item in ranked), 1.0)
    letters[start] = {"total": total, "strongest": ranked[0]["letter"], "outgoing": ranked}

for start, end, percent in [("E", "R", 19.6), ("Q", "U", 99.1), ("X", "P", 26.4)]:
    top = letters[start]["outgoing"][0]
    print(f"{start} → {top['letter']}: {top['probability']:.1%}")
    assert top["letter"] == end and round(100 * top["probability"], 1) == percent

E → R: 19.6%
Q → U: 99.1%
X → P: 26.4%


## Follow all 26 starts and measure convergence

Track the first visit to each state. A repeated state closes the cycle; its first occurrence gives the entry step. Count starting paths individually even after their positions coincide.

In [3]:
for start, record in letters.items():
    path = []
    current = start
    while current not in path:
        path.append(current)
        current = letters[current]["strongest"]
    entry = path.index(current)
    cycle = path[entry:]
    assert len(cycle) == 2 and set(cycle) == {"E", "R"}
    record.update(path=path + [current], cycle=cycle, entry_step=entry)

last_entry = max(record["entry_step"] for record in letters.values())
frames = []
positions = dict(zip(alphabet, alphabet))
for step in range(last_entry + 1):
    in_loop = sum(position in "ER" for position in positions.values())
    frames.append({"step": step, "positions": positions, "in_loop": in_loop})
    positions = {start: letters[position]["strongest"] for start, position in positions.items()}
assert frames[0]["in_loop"] == 2 and frames[-1]["in_loop"] == 26
print("Paths at E or R by step:", [frame["in_loop"] for frame in frames])

Paths at E or R by step: [2, 13, 18, 22, 26]


## Website handoff

Use frames for a simultaneous release on a fixed 26-node layout. Each position is keyed by starting letter. Pause on E after convergence and reveal the probability for R and the combined probability of all alternatives. Continue to the full graph with exactly one strongest outgoing edge per letter, preserving the layout and distinguishing both directions of E ↔ R.

Hover previews every observed outgoing connection of a letter. Include self-transitions. Omit zero-width edges; keep all 26 destinations in the table. Click or tap to pin, click again to clear. Hover temporarily overrides a pin; leaving restores it. Synchronize a letter dropdown and the adjacent ranked probability table with the graph. Clearing returns to the strongest-link overview. Use counts for exact values and round probabilities only for display. This is the one-letter-memory chapter; finish with the question of what changes when the model remembers two letters.

In [4]:
payload = {"source": "https://norvig.com/mayzner.html", "letters": letters, "frames": frames}
output = Path("letter_transitions.json")
output.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
assert json.loads(output.read_text()) == payload
print(output.name)

letter_transitions.json
